# Market Basket Analysis con FP-Growth — Instacart (a nivel de PASILLOS)

## Objetivo

En `notebooks/07_Market_Basket_Analysis.ipynb` construimos reglas de asociación
**a nivel de producto** (`product_id → product_id`): *"si el carrito tiene el
producto A, ¿qué producto C es probable que también quiera?"*.

En esta notebook vamos a construir el **mismo tipo de modelo**, pero mirando una
granularidad distinta: en vez de productos individuales, vamos a trabajar con
**pasillos (aisles)**. Instacart organiza sus ~50.000 productos en solo **134
pasillos** (por ejemplo: *fresh fruits*, *yogurt*, *chips pretzels*, *beer*...).

La pregunta que este modelo responde es: *"Si un cliente ya tiene productos del
pasillo A en el carrito, ¿de qué otro pasillo C es probable que también quiera
algo?"*. Por ejemplo: `{fresh vegetables} → {fresh herbs}` — quien compra
verduras frescas suele llevar también hierbas frescas.

## ¿Por qué a nivel de pasillo y no de producto?

Trabajar con pasillos en vez de productos tiene ventajas y desventajas que vale
la pena tener claras desde el arranque:

- **Menos "items" a minar**: 134 pasillos contra ~50.000 productos. La matriz de
  transacciones es mucho más chica y densa — casi no hace falta filtrar nada
  antes de minar (ver sección 5).
- **Patrones más robustos**: cada pasillo agrupa a muchos productos, así que el
  soporte de cada "item" es mucho mayor. Esto da reglas sostenidas por muchísima
  más evidencia estadística (menos riesgo de que sea puro ruido).
- **Menos específico**: perdemos el detalle de *qué producto exacto* recomendar
  (eso es lo que hace la NB07). Un pasillo es una categoría amplia: recomendar
  "pasillo yogurt" es menos accionable que recomendar "Icelandic Style Skyr
  Blueberry Non-fat Yogurt" puntual.
- **Lift más moderado**: como los pasillos son categorías amplias que casi todo
  el mundo visita en alguna medida, la independencia estadística "pura" es menos
  frecuente — vamos a ver lifts más cercanos a 1 que en la NB07 (donde había
  lifts de 70+ entre productos de nicho muy específicos).

En definitiva: el modelo de pasillos es un **complemento** al de productos, útil
para entender la estructura de compra a un nivel más alto (por ejemplo, para
decidir qué pasillos poner cerca en un layout de tienda, o para dar una primera
sugerencia amplia antes de bajar al detalle de producto).

## ¿Qué es una regla de asociación? (repaso rápido)

Una regla de asociación tiene la forma:

$$\text{antecedents} \rightarrow \text{consequents}$$

Acá, tanto `antecedents` como `consequents` van a ser conjuntos de **aisle_id**
(pasillos), no de productos. Por ejemplo: `{pasillo fresh vegetables} →
{pasillo fresh herbs}` significa *"las órdenes que llevan algo del pasillo de
verduras frescas, frecuentemente también llevan algo del pasillo de hierbas
frescas"*.

- **antecedents**: pasillos que ya están representados en el carrito (la
  "condición").
- **consequents**: pasillos que se recomienda explorar (la "consecuencia").

Igual que en la NB07, estas reglas se **descubren automáticamente** contando qué
combinaciones de pasillos aparecen juntas con frecuencia en el historial de
órdenes, en dos pasos: (1) minar itemsets frecuentes de pasillos con **FP-Growth**,
y (2) generar reglas a partir de esos itemsets, quedándonos con las
estadísticamente interesantes.

## Las métricas: support, confidence y lift

Repasemos las tres métricas con el mismo ejemplo numérico de la NB07, pero
pensando en pasillos en vez de productos.

Supongamos que tenemos **100 órdenes** en total, y de ellas:
- 40 órdenes tienen algo del pasillo **fresh vegetables**.
- 30 órdenes tienen algo del pasillo **fresh herbs**.
- 20 órdenes tienen algo de **ambos pasillos**.

### Support — ¿qué tan frecuente es esta combinación?

$$\text{support}(A \rightarrow B) = \frac{\text{ordenes que tocan A y B}}{\text{total de ordenes}}$$

En nuestro ejemplo: `support(fresh vegetables → fresh herbs) = 20 / 100 = 0.20`
(20%). El support mide **popularidad**: qué tan seguido aparece esa combinación
en el total de los datos.

### Confidence — de las que tocaron A, ¿cuántas tocaron también B?

$$\text{confidence}(A \rightarrow B) = \frac{\text{support}(A \rightarrow B)}{\text{support}(A)} = \frac{\text{ordenes con A y B}}{\text{ordenes con A}}$$

En nuestro ejemplo: `confidence(fresh vegetables → fresh herbs) = 20 / 40 = 0.50`
(50%). Es decir: *"de las órdenes que tocan el pasillo de verduras, el 50%
también toca el de hierbas"*. Es una probabilidad condicional
P(fresh herbs | fresh vegetables).

**Ojo con el mismo problema clásico que en la NB07**: si *fresh herbs* fuera un
pasillo que casi todo el mundo visita, la confidence sería alta simplemente por
esa popularidad, no por una relación real con *fresh vegetables*. Para eso está
el lift.

### Lift — ¿la relación es real o es pura casualidad estadística?

$$\text{lift}(A \rightarrow B) = \frac{\text{confidence}(A \rightarrow B)}{\text{support}(B)}$$

Sigamos el ejemplo: `support(fresh herbs) = 30/100 = 0.30`. Entonces
`lift(fresh vegetables → fresh herbs) = 0.50 / 0.30 = 1.67`.

Interpretación (igual que en la NB07):

- **lift = 1**: los pasillos son independientes. No hay relación real.
- **lift > 1**: aparecen juntos **más** de lo que el azar predeciría — la
  relación que nos interesa para recomendar.
- **lift < 1**: aparecen juntos **menos** de lo esperado (se "repelen").

**Por qué el lift importa todavía más a nivel pasillo**: como casi todos los
pasillos tienen soporte relativamente alto (categorías amplias), la confidence
por sí sola es aún menos confiable que a nivel producto. El lift sigue siendo la
métrica clave para separar "relación real" de "los dos pasillos son populares".

## ¿Por qué FP-Growth?

Igual que en la NB07: Apriori y FP-Growth encuentran exactamente los mismos
itemsets frecuentes (el resultado matemático es idéntico), pero FP-Growth
construye un árbol compacto (FP-tree) una sola vez y mina los patrones sin
generar candidatos explícitos, lo cual escala mejor. Acá con solo 134 pasillos
la diferencia de performance es casi anecdótica, pero mantenemos el mismo
algoritmo por consistencia con la NB07 y porque en datasets grandes sigue siendo
la elección correcta.

## 1. Imports y configuración

In [3]:
import time

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from mlxtend.frequent_patterns import association_rules, fpgrowth

### Parámetros del modelo

Los mismos cuatro parámetros que en la NB07, pero recalibrados para trabajar con
pasillos en vez de productos:

- **`MIN_SUPPORT`**: soporte mínimo para que un itemset de pasillos se considere
  "frecuente". En la NB07 usábamos `0.001` (0.1%) porque había ~50.000
  productos y la mayoría son de nicho. Acá, con solo 134 pasillos que agrupan
  muchísimos productos cada uno, el soporte individual de cada pasillo es mucho
  más alto — un `MIN_SUPPORT` tan bajo dejaría pasar casi cualquier combinación
  (poco selectivo). Por eso lo subimos a **`0.01`** (1% de las órdenes, ~32.000
  órdenes): igual de "generoso" en espíritu, pero calibrado a la escala nueva.
- **`MIN_CONFIDENCE`**: igual que en la NB07, `0.10` — confianza mínima para que
  una regla se considere lo bastante fiable como para generarse.
- **`MIN_LIFT`**: igual que en la NB07, `1.0` — descartamos lift ≤ 1 (sin valor
  real de recomendación).
- **`MAX_CONSEQUENT_SUPPORT`**: tope de popularidad para lo que el modelo puede
  *recomendar* (política "no recomendar lo obvio", sección 7.5). En la NB07 era
  `0.10` y solo excluía 2 productos (las dos bananas). Acá, como los pasillos
  son categorías amplias, pasillos como *fresh fruits* o *fresh vegetables*
  aparecen en una fracción mucho más grande de las órdenes — por eso subimos el
  tope a **`0.35`** (35%). Si lo dejáramos en `0.10`, terminaríamos bloqueando
  la enorme mayoría de los pasillos y el modelo casi no podría recomendar nada.

**Todos estos valores son tuneables** — son un punto de partida razonable, no
una verdad absoluta. Si después de correr la notebook ves muy pocas reglas (o
demasiadas, o que los consecuentes son siempre los mismos 3 pasillos), volvé a
esta celda y ajustá los números.

In [4]:
MIN_SUPPORT = 0.01        # 1% de las ordenes: el pasillo debe aparecer en al menos ~32.000 ordenes
MIN_CONFIDENCE = 0.15      # al menos 10% de fiabilidad condicional
MIN_LIFT = 1.3            # descartamos reglas con lift <= 1 (sin valor real de recomendacion)
MAX_CONSEQUENT_SUPPORT = 0.35  # no recomendamos pasillos presentes en mas del 35% de las ordenes
RANDOM_STATE = 42

DATA_DIR = '../data/raw/instacart'
OUTPUT_PATH = '../data/processed/association_rules_aisles.csv'

## 2. Carga de datos

Igual que en la NB07, usamos `order_products__prior.csv`: cada fila es *"este
producto estuvo en esta orden"*. Cargamos solo `order_id` y `product_id`, con
`dtype int32` para ahorrar memoria (32,4 millones de filas).

La diferencia clave respecto a la NB07 es que acá **no nos importa el producto
en sí, sino su pasillo**. Para eso:

1. Cargamos `products.csv` (`product_id, product_name, aisle_id,
   department_id`) y armamos `product_to_aisle`: un diccionario
   `product_id → aisle_id`.
2. Cargamos `aisles.csv` (`aisle_id, aisle`) y armamos `id_to_name`: un
   diccionario `aisle_id → nombre del pasillo` (el equivalente de `id_to_name`
   de la NB07, pero para pasillos en vez de productos).
3. Traducimos cada línea de `order_products` de `product_id` a `aisle_id`.
4. **Deduplicamos** por `(order_id, aisle_id)`: si una orden tiene 3 productos
   distintos del pasillo *yogurt*, para nosotros esa orden "toca" el pasillo
   *yogurt* una sola vez — no nos interesa contar cuántos productos de ese
   pasillo llevó, solo si estuvo presente o no (es la misma idea de codificación
   one-hot que usa la NB07 a nivel producto).

In [5]:
t0 = time.time()
order_products = pd.read_csv(
    f'{DATA_DIR}/order_products__prior.csv',
    usecols=['order_id', 'product_id'],
    dtype={'order_id': 'int32', 'product_id': 'int32'},
)
products = pd.read_csv(f'{DATA_DIR}/products.csv')
aisles = pd.read_csv(f'{DATA_DIR}/aisles.csv')

product_to_aisle = dict(zip(products['product_id'], products['aisle_id']))
id_to_name = dict(zip(aisles['aisle_id'], aisles['aisle']))

print(f'order_products: {order_products.shape[0]:,} filas, {order_products.shape[1]} columnas')
print(f'productos unicos: {order_products["product_id"].nunique():,}')
print(f'pasillos totales (aisles.csv): {len(aisles):,}')
print(f'tiempo de carga: {time.time() - t0:.1f}s')

order_products: 32,434,489 filas, 2 columnas
productos unicos: 49,677
pasillos totales (aisles.csv): 134
tiempo de carga: 5.4s


In [6]:
t0 = time.time()
order_aisles = order_products.copy()
order_aisles['aisle_id'] = order_aisles['product_id'].map(product_to_aisle).astype('int32')
order_aisles = order_aisles[['order_id', 'aisle_id']].drop_duplicates()

print(f'filas antes de deduplicar por (order_id, aisle_id): {len(order_products):,}')
print(f'filas despues de deduplicar: {len(order_aisles):,}')
print(f'ordenes unicas: {order_aisles["order_id"].nunique():,}')
print(f'pasillos unicos presentes en el historial: {order_aisles["aisle_id"].nunique():,}')
print(f'tiempo: {time.time() - t0:.1f}s')

filas antes de deduplicar por (order_id, aisle_id): 32,434,489
filas despues de deduplicar: 23,338,453
ordenes unicas: 3,214,874
pasillos unicos presentes en el historial: 134
tiempo: 4.0s


## 3. Codificación one-hot de las transacciones (la "canasta" de pasillos)

Igual que en la NB07: `mlxtend` necesita una tabla donde cada **fila es una
orden** y cada **columna es un pasillo**, con `True`/`False` según si esa orden
tocó ese pasillo.

Acá la cuenta de tamaño es mucho más chica que en la NB07:

$$3.214.874 \text{ ordenes} \times 134 \text{ pasillos} \approx 431 \text{ millones de celdas}$$

Esto son apenas ~431 MB si lo guardáramos denso (contra los ~160 GB a nivel
producto) — **perfectamente manejable incluso sin sparse**. Aun así, seguimos
usando `scipy.sparse.csr_matrix` por consistencia con la NB07 y porque la
construcción vía códigos categóricos es igual de rápida.

In [7]:
t0 = time.time()
order_cat = order_aisles['order_id'].astype('category')
aisle_cat = order_aisles['aisle_id'].astype('category')

row_idx = order_cat.cat.codes.to_numpy()          # que fila (orden) es cada linea
col_idx = aisle_cat.cat.codes.to_numpy()          # que columna (pasillo) es cada linea
n_orders = len(order_cat.cat.categories)
all_aisle_ids = aisle_cat.cat.categories.to_numpy()  # aisle_id real detras de cada columna

values = np.ones(len(order_aisles), dtype=bool)
basket_matrix = csr_matrix(
    (values, (row_idx, col_idx)),
    shape=(n_orders, len(all_aisle_ids)),
    dtype=bool,
)

print(f'matriz sparse: {basket_matrix.shape[0]:,} ordenes x {basket_matrix.shape[1]:,} pasillos')
print(f'valores True (presente): {basket_matrix.nnz:,} ({basket_matrix.nnz / basket_matrix.shape[0]:.1f} pasillos por orden en promedio)')
densidad = basket_matrix.nnz / (basket_matrix.shape[0] * basket_matrix.shape[1])
print(f'densidad: {densidad*100:.2f}% de las celdas son True')
print(f'tiempo: {time.time() - t0:.1f}s')

matriz sparse: 3,214,874 ordenes x 134 pasillos
valores True (presente): 23,338,453 (7.3 pasillos por orden en promedio)
densidad: 5.42% de las celdas son True
tiempo: 0.9s


Como era de esperar, la densidad acá es **mucho mayor** que en la NB07
(donde era apenas 0,02%): con solo 134 columnas, cada orden "enciende" un
porcentaje mucho más grande de ellas. Esto es bueno para nuestro análisis — más
densidad significa más evidencia estadística por pasillo, y reglas más
robustas.

## 4. Filtrar pasillos poco frecuentes antes de minar

En la NB07 este paso era **obligatorio** para no quedarse sin memoria (mlxtend
densifica internamente la matriz completa antes de construir el FP-tree, y con
49.677 productos eso son ~160 GB). Acá, con 134 pasillos, ese mismo paso interno
densifica una matriz de apenas `3.214.874 × 134 ≈ 431` millones de celdas — nada
que ponga en riesgo la memoria de una notebook normal.

Igual mantenemos el filtro por **consistencia de estilo** con la NB07 y porque
la propiedad matemática detrás sigue siendo válida: por
**anti-monotonicidad** (*downward closure*), si un pasillo no llega al soporte
mínimo por sí solo, ningún itemset que lo contenga puede llegar tampoco — así
que es matemáticamente seguro descartarlo de entrada. A esta escala, esperamos
que el filtro descarte pocos o ningún pasillo (la mayoría de los 134 pasillos de
Instacart tienen soporte individual bastante alto).

In [8]:
item_support = np.asarray(basket_matrix.sum(axis=0)).ravel() / n_orders
keep_mask = item_support >= MIN_SUPPORT
kept_aisle_ids = all_aisle_ids[keep_mask]
basket_matrix_reduced = basket_matrix[:, keep_mask]

peak_ram_gb = n_orders * len(kept_aisle_ids) / 1e9
print(f'pasillos que sobreviven al filtro: {len(kept_aisle_ids):,} de {len(all_aisle_ids):,}')
print(f'pico de RAM estimado en el paso de FP-Growth: ~{peak_ram_gb:.2f} GB')

# mlxtend exige nombres de columna string (no puede haber columnas enteras en formato sparse)
basket = pd.DataFrame.sparse.from_spmatrix(
    basket_matrix_reduced,
    columns=[str(a) for a in kept_aisle_ids],
)
basket.shape

pasillos que sobreviven al filtro: 96 de 134
pico de RAM estimado en el paso de FP-Growth: ~0.31 GB


(3214874, 96)

## 5. Minar itemsets frecuentes con FP-Growth

`fpgrowth` recorre la matriz de transacciones y devuelve todas las combinaciones
de pasillos (`itemsets`) cuyo `support` es mayor o igual a `MIN_SUPPORT`.
Esperamos que corra en segundos (contra los ~72s de la NB07): 134 columnas es
una fracción minúscula de las 1.772 con las que trabajaba FP-Growth en la
NB07.

In [9]:
t0 = time.time()
frequent_itemsets = fpgrowth(basket, min_support=MIN_SUPPORT, use_colnames=True)
elapsed = time.time() - t0

print(f'itemsets frecuentes encontrados: {len(frequent_itemsets):,}')
print(f'tiempo de FP-Growth: {elapsed:.1f}s')
frequent_itemsets.sort_values('support', ascending=False).head(10)

itemsets frecuentes encontrados: 2,897
tiempo de FP-Growth: 73.6s


,support,itemsets
20,0.557027,frozenset({24})
0,0.444071,frozenset({83})
1,0.366808,frozenset({123})
96,0.317762,"frozenset({83, 24})"
98,0.270270,"frozenset({123, 24})"
8,0.263488,frozenset({120})
21,0.244485,frozenset({84})
97,0.234454,"frozenset({123, 83})"
22,0.229527,frozenset({21})
23,0.191012,frozenset({115})


## 6. Generar las reglas de asociación

`association_rules` toma los itemsets frecuentes y arma todas las reglas
posibles `antecedents → consequents` a partir de sus subconjuntos, calculando
`confidence` y `lift` para cada una. Filtramos por `MIN_CONFIDENCE` al
generarlas, y después por `MIN_LIFT` (nos quedamos solo con relaciones reales,
no casualidades).

El resultado final tiene exactamente las columnas que buscamos:
`['antecedents', 'consequents', 'support', 'confidence', 'lift']`, ordenado por
`lift` descendente (las relaciones más fuertes primero). La única diferencia
con la NB07 es que acá `antecedents`/`consequents` son conjuntos de
**`aisle_id`**, no de `product_id`.

In [10]:
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=MIN_CONFIDENCE)
rules = rules[rules['lift'] >= MIN_LIFT].copy()

# de vuelta a aisle_id como entero (mlxtend los devuelve como string porque asi los mandamos)
rules['antecedents'] = rules['antecedents'].apply(lambda s: frozenset(int(i) for i in s))
rules['consequents'] = rules['consequents'].apply(lambda s: frozenset(int(i) for i in s))

rules = (
    rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
    .sort_values('lift', ascending=False)
    .reset_index(drop=True)
)

print(f'reglas generadas: {len(rules):,}')
rules.head(10)

reglas generadas: 11,021


,antecedents,consequents,support,confidence,lift
0,"frozenset({24, 131})","frozenset({9, 83})",0.010046,0.207450,5.116844
1,"frozenset({9, 83})","frozenset({24, 131})",0.010046,0.247792,5.116844
2,"frozenset({83, 131})","frozenset({24, 9})",0.010046,0.211515,5.007825
3,"frozenset({24, 9})","frozenset({83, 131})",0.010046,0.237852,5.007825
4,"frozenset({83, 59})","frozenset({24, 81})",0.011693,0.234359,4.718410
5,"frozenset({24, 81})","frozenset({83, 59})",0.011693,0.235415,4.718410
6,"frozenset({123, 131})",frozenset({9}),0.010534,0.288676,4.657073
7,frozenset({9}),"frozenset({123, 131})",0.010534,0.169933,4.657073
8,"frozenset({9, 123})",frozenset({131}),0.010534,0.323738,4.582644
9,"frozenset({24, 9, 83})",frozenset({131}),0.010046,0.323232,4.575474


### Leyendo una regla real

Traduzcamos a lenguaje humano las 3 reglas con mayor lift, usando los mismos
números de la tabla (mismo formato que en la NB07, pero con nombres de
pasillo):

In [11]:
for _, r in rules.head(3).iterrows():
    ante = ', '.join(id_to_name[a] for a in r['antecedents'])
    cons = ', '.join(id_to_name[a] for a in r['consequents'])
    print(f"Si el carrito toca el/los pasillo(s) [{ante}]:")
    print(f"  -> el {r['confidence']*100:.1f}% de esas ordenes tambien toco [{cons}]")
    print(f"  -> eso es {r['lift']:.2f}x mas probable que si fuera puro azar (lift={r['lift']:.2f})")
    print(f"  -> esta combinacion aparece en el {r['support']*100:.2f}% de TODAS las ordenes (support={r['support']:.4f})")
    print()

Si el carrito toca el/los pasillo(s) [fresh fruits, dry pasta]:
  -> el 20.7% de esas ordenes tambien toco [pasta sauce, fresh vegetables]
  -> eso es 5.12x mas probable que si fuera puro azar (lift=5.12)
  -> esta combinacion aparece en el 1.00% de TODAS las ordenes (support=0.0100)

Si el carrito toca el/los pasillo(s) [pasta sauce, fresh vegetables]:
  -> el 24.8% de esas ordenes tambien toco [fresh fruits, dry pasta]
  -> eso es 5.12x mas probable que si fuera puro azar (lift=5.12)
  -> esta combinacion aparece en el 1.00% de TODAS las ordenes (support=0.0100)

Si el carrito toca el/los pasillo(s) [fresh vegetables, dry pasta]:
  -> el 21.2% de esas ordenes tambien toco [fresh fruits, pasta sauce]
  -> eso es 5.01x mas probable que si fuera puro azar (lift=5.01)
  -> esta combinacion aparece en el 1.00% de TODAS las ordenes (support=0.0100)



## 7. Versión legible con nombres de pasillo

Guardamos también una copia de las reglas con los nombres de pasillo en texto
plano, para poder leerlas de un vistazo sin ir a buscar cada `aisle_id`.

In [12]:
rules_readable = rules.copy()
rules_readable['antecedents_names'] = rules_readable['antecedents'].apply(
    lambda s: [id_to_name.get(a, f'?{a}') for a in s]
)
rules_readable['consequents_names'] = rules_readable['consequents'].apply(
    lambda s: [id_to_name.get(a, f'?{a}') for a in s]
)
rules_readable[['antecedents_names', 'consequents_names', 'support', 'confidence', 'lift']].head(50)

,antecedents_names,consequents_names,support,confidence,lift
0,"[fresh fruits, dry pasta]","[pasta sauce, fresh vegetables]",0.010046,0.207450,5.116844
1,"[pasta sauce, fresh vegetables]","[fresh fruits, dry pasta]",0.010046,0.247792,5.116844
2,"[fresh vegetables, dry pasta]","[fresh fruits, pasta sauce]",0.010046,0.211515,5.007825
3,"[fresh fruits, pasta sauce]","[fresh vegetables, dry pasta]",0.010046,0.237852,5.007825
4,"[fresh vegetables, canned meals beans]","[fresh fruits, canned jarred vegetables]",0.011693,0.234359,4.718410
5,"[fresh fruits, canned jarred vegetables]","[fresh vegetables, canned meals beans]",0.011693,0.235415,4.718410
6,"[packaged vegetables fruits, dry pasta]",[pasta sauce],0.010534,0.288676,4.657073
7,[pasta sauce],"[packaged vegetables fruits, dry pasta]",0.010534,0.169933,4.657073
8,"[pasta sauce, packaged vegetables fruits]",[dry pasta],0.010534,0.323738,4.582644
9,"[fresh fruits, pasta sauce, fresh vegetables]",[dry pasta],0.010046,0.323232,4.575474


## 7.5 Política de recomendación: no recomendar lo obvio

En la NB07, banana era el producto ultra-popular que había que filtrar. A nivel
pasillo, el equivalente son categorías tan amplias que casi cualquier orden las
toca — típicamente **fresh fruits** y **fresh vegetables**. Vale la pena
mirar la tabla de soporte por pasillo antes de decidir el filtro.

**No es un bug — es la firma de un pasillo ultra-popular.** Igual que con la
banana, estos pasillos van a co-ocurrir con casi cualquier otro con confidence
alta, simplemente porque casi todo el mundo pasa por ahí. El lift de esas reglas
suele ser más bajo que el de reglas entre pasillos más específicos — la métrica
ya nos avisa "esto es popularidad, no una relación fuerte" — pero con
`MIN_LIFT=1.0` igual las dejamos pasar en la tabla de reglas.

**Decisión de diseño (igual que en la NB07):** no tocamos la tabla `rules` (la
dejamos intacta para no perder información de análisis). Aplicamos la política
"no recomendar lo obvio" en la capa de recomendación: `recommend_from_cart` va a
descartar cualquier pasillo candidato cuyo soporte global supere
`MAX_CONSEQUENT_SUPPORT`. No tiene sentido "recomendar" el pasillo de frutas
frescas a alguien que ya lo visita en la enorme mayoría de sus compras.

In [13]:
# reusamos el item_support ya calculado en la seccion 4 (soporte global de CADA pasillo,
# no solo los que sobrevivieron al filtro de mineria)
aisle_support = pd.Series(item_support, index=all_aisle_ids)

print('soporte global de los 10 pasillos mas frecuentes:')
top_support = aisle_support.sort_values(ascending=False).head(10)
for aid, supp in top_support.items():
    print(f'  {aid:>4}  {supp*100:5.1f}%  {id_to_name[aid]}')

print()
blocked = aisle_support[aisle_support > MAX_CONSEQUENT_SUPPORT].sort_values(ascending=False)
print(f'con MAX_CONSEQUENT_SUPPORT={MAX_CONSEQUENT_SUPPORT}, se excluyen {len(blocked)} pasillos como recomendacion:')
for aid, supp in blocked.items():
    print(f'  {aid:>4}  {supp*100:5.1f}%  {id_to_name[aid]}')

soporte global de los 10 pasillos mas frecuentes:
    24   55.7%  fresh fruits
    83   44.4%  fresh vegetables
   123   36.7%  packaged vegetables fruits
   120   26.3%  yogurt
    84   24.4%  milk
    21   23.0%  packaged cheese
   115   19.1%  water seltzer sparkling water
    91   17.0%  soy lactosefree
   107   16.7%  chips pretzels
   112   16.4%  bread

con MAX_CONSEQUENT_SUPPORT=0.35, se excluyen 3 pasillos como recomendacion:
    24   55.7%  fresh fruits
    83   44.4%  fresh vegetables
   123   36.7%  packaged vegetables fruits


## 8. Función recomendadora: de un carrito de productos a una lista de pasillos sugeridos

`recommend_from_cart` recibe una lista de **`product_id`** (el carrito real del
cliente, tal como llegaría de una app de compras) y devuelve **pasillos**
sugeridos. La lógica:

1. Traducimos los `product_id` del carrito a los `aisle_id` que representan
   (vía `product_to_aisle`) — esto define el conjunto de pasillos "ya tocados".
2. Nos quedamos solo con las reglas cuyo `antecedents` sea un **subconjunto** de
   esos pasillos.
3. Descartamos de las sugerencias los pasillos que **ya están** representados
   en el carrito (no tiene sentido recomendar un pasillo que el cliente ya está
   visitando).
4. Descartamos los pasillos cuyo soporte global supera
   `MAX_CONSEQUENT_SUPPORT` (política de la sección 7.5).
5. Si varias reglas distintas sugieren el mismo pasillo, nos quedamos con la de
   mejor `lift` (y `confidence` como desempate).
6. Ordenamos todo por `lift` descendente y devolvemos el top-N.

In [14]:
def recommend_from_cart(cart_product_ids, rules_df, top_n=10, max_consequent_support=MAX_CONSEQUENT_SUPPORT):
    """Recomienda PASILLOS a partir de un carrito de productos, usando las reglas de asociacion.

    Parameters
    ----------
    cart_product_ids : list[int]
        product_id de los productos que ya estan en el carrito (no aisle_id).
    rules_df : pd.DataFrame
        reglas con columnas ['antecedents', 'consequents', 'support', 'confidence', 'lift'],
        donde antecedents/consequents son conjuntos de aisle_id.
    top_n : int
        cantidad maxima de pasillos a devolver.
    max_consequent_support : float or None
        no se recomienda ningun pasillo cuyo soporte global (popularidad) supere este valor
        (ver seccion 7.5: evita recomendar siempre fresh fruits/fresh vegetables). Pasa None
        para desactivar este filtro y ver el comportamiento sin de-sesgar.
    """
    cart_aisles = {product_to_aisle[pid] for pid in cart_product_ids if pid in product_to_aisle}

    # regla aplicable = todos sus antecedentes (pasillos) ya estan representados en el carrito
    applicable = rules_df[rules_df['antecedents'].apply(lambda a: a.issubset(cart_aisles))]

    best_per_aisle = {}
    for _, r in applicable.iterrows():
        for aid in r['consequents']:
            if aid in cart_aisles:
                continue  # no recomendamos un pasillo que ya esta representado en el carrito
            if max_consequent_support is not None and aisle_support.get(aid, 0) > max_consequent_support:
                continue  # pasillo demasiado popular: no aporta valor recomendarlo (seccion 7.5)
            candidate = (r['lift'], r['confidence'])
            if aid not in best_per_aisle or candidate > best_per_aisle[aid][:2]:
                best_per_aisle[aid] = (r['lift'], r['confidence'], aid)

    if not best_per_aisle:
        return pd.DataFrame(columns=['aisle_id', 'aisle_name', 'confidence', 'lift'])

    recs = pd.DataFrame(
        [{'aisle_id': aid, 'lift': lift, 'confidence': conf} for lift, conf, aid in best_per_aisle.values()]
    )
    recs['aisle_name'] = recs['aisle_id'].map(id_to_name)
    recs = recs.sort_values(['lift', 'confidence'], ascending=False).head(top_n).reset_index(drop=True)
    return recs[['aisle_id', 'aisle_name', 'confidence', 'lift']]

## 9. Prueba manual: armamos un "carrito" con unos pocos product_id y validamos

Vamos a simular carritos por `product_id` (como llegaría un carrito real) y ver
qué **pasillos** recomienda el modelo. Usamos el mismo primer carrito de la
NB07 (espinaca orgánica + palta orgánica) para poder comparar el nivel de
detalle entre ambos enfoques.

Podés cambiar los `product_id` de la lista `cart` por cualquier otro (podés
buscarlos en `products.csv` filtrando por `product_name`) y volver a correr la
celda para ver cómo cambian las recomendaciones de pasillo.

In [15]:
cart = [21903, 47209]  # Organic Baby Spinach, Organic Hass Avocado
print('Carrito actual (productos):')
for pid in cart:
    aid = product_to_aisle[pid]
    print(f'  - {pid}: {products.loc[products.product_id == pid, "product_name"].values[0]}  (pasillo {aid}: {id_to_name[aid]})')

print()
print('Recomendaciones SIN de-sesgar (max_consequent_support=None):')
display(recommend_from_cart(cart, rules, max_consequent_support=None))

print()
print(f'Recomendaciones CON de-sesgar (max_consequent_support={MAX_CONSEQUENT_SUPPORT}, el default):')
recommend_from_cart(cart, rules)

Carrito actual (productos):
  - 21903: Organic Baby Spinach  (pasillo 123: packaged vegetables fruits)
  - 47209: Organic Hass Avocado  (pasillo 24: fresh fruits)

Recomendaciones SIN de-sesgar (max_consequent_support=None):


,aisle_id,aisle_name,confidence,lift
0,83,fresh vegetables,0.154695,1.985680
1,116,frozen produce,0.154695,1.985680
2,120,yogurt,0.280317,1.946080
3,91,soy lactosefree,0.183619,1.941606
4,86,eggs,0.158012,1.868650
5,21,packaged cheese,0.250023,1.853776
6,112,bread,0.167840,1.843821
7,84,milk,0.230339,1.841406
8,16,fresh herbs,0.160051,1.713072
9,67,fresh dips tapenades,0.160292,1.636400



Recomendaciones CON de-sesgar (max_consequent_support=0.35, el default):


,aisle_id,aisle_name,confidence,lift
0,116,frozen produce,0.154695,1.985680
1,120,yogurt,0.280317,1.946080
2,91,soy lactosefree,0.183619,1.941606
3,86,eggs,0.158012,1.868650
4,21,packaged cheese,0.250023,1.853776
5,112,bread,0.167840,1.843821
6,84,milk,0.230339,1.841406
7,16,fresh herbs,0.160051,1.713072
8,67,fresh dips tapenades,0.160292,1.636400
9,96,lunch meat,0.158731,1.527158


Comparando ambas tablas: si algún pasillo ultra-popular (como *fresh
fruits* o *fresh vegetables*) aparecía en la versión sin de-sesgar, debería
desaparecer en la versión con de-sesgar — quedó excluido por superar
`MAX_CONSEQUENT_SUPPORT`. Ese es el mismo trade-off explicado en la sección 7.5,
ahora en acción sobre un caso real.

Probá con otro carrito para ver cómo cambia la recomendación de pasillo:

In [16]:
# products[products['product_name'].str.contains('Strawberr', case=False)]

cart_2 = [49610, 132]  # 100% Lactose Free Fat Free Milk, Yogurt Strawberry Pomegranate
print('Carrito actual (productos):')
for pid in cart_2:
    aid = product_to_aisle[pid]
    print(f'  - {pid}: {products.loc[products.product_id == pid, "product_name"].values[0]}  (pasillo {aid}: {id_to_name[aid]})')

print()
print('Recomendaciones:')
recommend_from_cart(cart_2, rules)

Carrito actual (productos):
  - 49610: 100% Lactose Free Fat Free Milk  (pasillo 91: soy lactosefree)
  - 132: Yogurt Strawberry Pomegranate  (pasillo 120: yogurt)

Recomendaciones:


,aisle_id,aisle_name,confidence,lift
0,21,packaged cheese,0.209332,2.331916
1,116,frozen produce,0.193845,2.168286
2,31,refrigerated,0.175755,2.037560
3,84,milk,0.177075,2.021419
4,112,bread,0.223984,1.993311
5,107,chips pretzels,0.204849,1.945275
6,121,cereal,0.179694,1.943088
7,86,eggs,0.182108,1.905579
8,67,fresh dips tapenades,0.179656,1.834087
9,115,water seltzer sparkling water,0.193166,1.757153


**¿Qué pasa si el carrito no tiene ninguna regla aplicable?** Igual que en
la NB07, si armamos un carrito cuyos pasillos quedaron fuera del filtro de
soporte mínimo (poco probable a este nivel, pero posible si `MIN_SUPPORT` es
alto), `recommend_from_cart` devuelve una tabla **vacía** — no es un error, es
el modelo diciendo honestamente "no tengo evidencia suficiente para recomendar
nada acá" (efecto de *cold-start*, ver conclusiones).

## 10. Guardar las reglas (opcional)

Guardamos las reglas calculadas en `data/processed/` para poder reutilizarlas
sin tener que recalcular todo el pipeline de nuevo.

In [17]:
rules.to_csv(OUTPUT_PATH, index=False)
print(f'reglas guardadas en {OUTPUT_PATH}')

reglas guardadas en ../data/processed/association_rules_aisles.csv


## 11. Exportar el modelo para uso fuera del notebook

Igual que en la NB07, empaquetamos todo lo que `recommend_from_cart` necesita
en un único artefacto `.joblib` autocontenido: reglas, soporte global por
pasillo, nombres de pasillo, y el mapeo `product_to_aisle` (imprescindible acá
porque el carrito de entrada sigue siendo por `product_id`, no por `aisle_id`).

**Dónde vive el código reusable:**
[`src/aisle_recommender.py`](../src/aisle_recommender.py), clase
`AisleRecommender`. Misma lógica que `recommend_from_cart` de la sección 8 — la
duplicación es intencional, igual que en la NB07: acá arriba la función queda
para leerla paso a paso mientras aprendés; `src/aisle_recommender.py` es la
versión que se importa desde otro lado. Devuelve una lista de dicts
`{'aisle_id', 'aisle_name', 'rank'}`.

In [18]:
import joblib
from datetime import datetime, timezone

MODEL_PATH = '../models/mba_aisle_recommender.joblib'

artifact = {
    'rules': rules,
    'aisle_support': aisle_support,
    'id_to_name': id_to_name,
    'product_to_aisle': product_to_aisle,
    'max_consequent_support': MAX_CONSEQUENT_SUPPORT,
    'metadata': {
        'generated_at': datetime.now(timezone.utc).isoformat(),
        'min_support': MIN_SUPPORT,
        'min_confidence': MIN_CONFIDENCE,
        'min_lift': MIN_LIFT,
        'n_rules': len(rules),
        'source_notebook': '08_Market_Basket_Analysis_Aisles.ipynb',
    },
}

joblib.dump(artifact, MODEL_PATH)
print(f'modelo exportado a {MODEL_PATH}')
print(f"reglas empaquetadas: {artifact['metadata']['n_rules']:,}")

modelo exportado a ../models/mba_aisle_recommender.joblib
reglas empaquetadas: 11,021


### Probando que el modelo exportado funciona standalone

Importamos `MBAAisleRecommender` desde `src/` (afuera del notebook, como lo haría
cualquier otro script) y comparamos, para los mismos dos carritos que ya usamos
en la sección 9, que el **orden de `aisle_id`** coincide exactamente con
`recommend_from_cart`.

In [19]:
import sys
sys.path.append('../src')
from mba_aisle_recommender import MBA_Aisle_Recommender

model = MBA_Aisle_Recommender.load()  # sin argumentos: encuentra models/mba_aisle_recommender.joblib solo
print('modelo cargado, metadata:', model.metadata)

modelo cargado, metadata: {'generated_at': '2026-07-02T22:28:24.825825+00:00', 'min_support': 0.01, 'min_confidence': 0.15, 'min_lift': 1.3, 'n_rules': 11021, 'source_notebook': '08_Market_Basket_Analysis_Aisles.ipynb'}


In [20]:
for test_cart in [cart, cart_2]:
    ids_notebook = recommend_from_cart(test_cart, rules)['aisle_id'].tolist()
    ids_module = [r['aisle_id'] for r in model.recommend(test_cart)]

    nombres_carrito = [products.loc[products.product_id == pid, 'product_name'].values[0] for pid in test_cart]
    print('carrito:', nombres_carrito)
    print('  recommend_from_cart  :', ids_notebook)
    print('  MBA_Aisle_Recommender.recommend:', ids_module)
    print('  mismo orden de aisle_id:', ids_notebook == ids_module)
    print()

print('salida de MBA_Aisle_Recommender.recommend (formato final, exportable):')
model.recommend(cart)

carrito: ['Organic Baby Spinach', 'Organic Hass Avocado']
  recommend_from_cart  : [116, 120, 91, 86, 21, 112, 84, 16, 67, 96]
  MBA_Aisle_Recommender.recommend: [116, 120, 91, 86, 21, 112, 84, 16, 67, 96]
  mismo orden de aisle_id: True

carrito: ['100% Lactose Free Fat Free Milk', 'Yogurt Strawberry Pomegranate']
  recommend_from_cart  : [21, 116, 31, 84, 112, 107, 121, 86, 67, 115]
  MBA_Aisle_Recommender.recommend: [21, 116, 31, 84, 112, 107, 121, 86, 67, 115]
  mismo orden de aisle_id: True

salida de MBA_Aisle_Recommender.recommend (formato final, exportable):


[{'aisle_id': 116, 'aisle_name': 'frozen produce', 'rank': 1},
 {'aisle_id': 120, 'aisle_name': 'yogurt', 'rank': 2},
 {'aisle_id': 91, 'aisle_name': 'soy lactosefree', 'rank': 3},
 {'aisle_id': 86, 'aisle_name': 'eggs', 'rank': 4},
 {'aisle_id': 21, 'aisle_name': 'packaged cheese', 'rank': 5},
 {'aisle_id': 112, 'aisle_name': 'bread', 'rank': 6},
 {'aisle_id': 84, 'aisle_name': 'milk', 'rank': 7},
 {'aisle_id': 16, 'aisle_name': 'fresh herbs', 'rank': 8},
 {'aisle_id': 67, 'aisle_name': 'fresh dips tapenades', 'rank': 9},
 {'aisle_id': 96, 'aisle_name': 'lunch meat', 'rank': 10}]

## Conclusiones

**Cómo leer las métricas (igual que a nivel producto):**
- **support** = qué tan frecuente es la combinación de pasillos en el total de
  órdenes (popularidad).
- **confidence** = de las órdenes que tocan el/los pasillo(s) antecedente(s),
  qué porcentaje también toca el/los pasillo(s) consecuente(s).
- **lift** = cuánto más probable es la combinación comparado con el azar puro.
  `lift > 1` es lo que buscamos para recomendar.

**Diferencias observadas frente a la NB07 (nivel producto):**
- Muchas menos "columnas" para minar (134 pasillos vs ~50.000 productos), así
  que el filtro de soporte mínimo (sección 4) es casi irrelevante acá, mientras
  que en la NB07 era un paso obligatorio para no quedarse sin memoria.
- Los pasillos tienen soporte individual mucho más alto, así que en general los
  **lifts son más moderados** que a nivel producto (ahí veíamos lifts de 70+
  entre productos de nicho muy específico; acá es esperable ver números mucho
  más chicos, típicamente entre 1 y unos pocos dígitos).
- El modelo de pasillo es **menos específico pero más robusto**: cada regla está
  sostenida por muchísima más evidencia (más órdenes involucradas), a costa de
  perder el detalle de qué producto puntual recomendar.

**Cómo validamos el modelo en esta notebook (validación "simple", igual que en
la NB07):**
1. Las métricas intrínsecas de las reglas (support/confidence/lift).
2. La prueba manual armando carritos de productos y revisando si los pasillos
   recomendados tienen sentido de negocio.

**Limitaciones a tener en cuenta:**
- No mide qué tan bien predice sobre compras futuras no vistas — solo qué tan
  fuertes son los patrones dentro del propio historial usado para entrenar.
- Hay el mismo efecto de **cold-start**: un pasillo cuyo soporte individual está
  por debajo de `MIN_SUPPORT` no aparece en ninguna regla.
- Al ser un modelo agregado por categoría, dos pasillos pueden tener lift alto
  simplemente por cómo Instacart definió esa taxonomía (por ejemplo, si dos
  pasillos parecidos en realidad deberían ser uno solo) — vale la pena mirar la
  tabla legible (sección 7) con ojo crítico.

**Próximo paso natural**: igual que en la NB07, una evaluación offline con
hold-out (entrenar con `order_products__prior.csv`, evaluar escondiendo un
pasillo por orden de `order_products__train.csv` y midiendo hit-rate /
precision@k), y comparar qué tan seguido el modelo de pasillos y el de
productos coinciden en su sugerencia de "categoría" recomendada.